In [ ]:
# INQUIRE toxicity-driven prioritization - Part 1: Preprocessing
"""
Reads the raw merged tox vs. dust data and the feature metadata, cleans and checks them, 
preprocesses the dust features (detection-frequency filter, below-LOD handling, log10, scaling) 
and the tox endpoints, and saves the inputs for Part 2 (modeling) to `OUT_DIR` with a `yymmdd_` prefix.
Run all cells. The last cell writes an Excel file describing all preprocessing steps.
"""


In [ ]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Imports & setup
import warnings
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import random
from collections import Counter # used by strip_column_names()
from scipy.stats import skew


# Logging setup (equivalent to logger + appender_tee)
# log_file = f"inquire_tip_pt1_preprocessing_{datetime.now().strftime('%y%m%d')}.log"
log_file = f"inquire_tip_pt1_preprocessing_{datetime.now().strftime('%y%m%d_%H%M%S')}.log" # date + time: a new run never overwrites the log of an earlier run

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler(log_file, mode="w"),
        logging.StreamHandler()
    ],
    force=True # re-applies the setup on re-runs (without it, the log file is not re-created after %reset)
)

# Helper functions
def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

# Validation helpers
def check_file(path_obj, label):
    if not path_obj.exists():
        raise FileNotFoundError(f" - {label} missing: {path_obj}")
    logging.info(f" - Found file: {label}")

def _read_header_only(path: Path):
    suffix = path.suffix.lower()
    if suffix in [".csv", ".tsv"]:
        sep = "\t" if suffix == ".tsv" else ","
        df = pd.read_csv(path, nrows=0, sep=sep)
        return list(df.columns)
    elif suffix in [".xlsx", ".xls"]:
        # Excel: only reads header row
        df = pd.read_excel(path, nrows=0)
        return list(df.columns)
    else:
        raise ValueError(f" - Unsupported file type: {suffix}")

def check_cols_before_load(path, expected_cols, dataset_name="dataset"):
    path = Path(path)
    # clear message if the file itself is missing
    if not path.exists():
        raise FileNotFoundError(f" - [{dataset_name}] file not found: {path}")
    try:
        cols = _read_header_only(path)
    except Exception as e:
        raise RuntimeError(f" - [{dataset_name}] failed to read header: {e}")
    # missing = [c for c in expected_cols if c not in cols]
    missing = [c for c in expected_cols if c not in [str(x).strip() for x in cols]] # ignore stray spaces in headers
    if missing:
        raise ValueError(
            f" - [{dataset_name}] missing columns: {missing}\n"
            f" - Available columns: {cols}"
        )
    logging.info(f" - [{dataset_name}] column check passed")
    return True

def safe_read_csv(path_obj, label, **kwargs):
    if "low_memory" not in kwargs:
        kwargs["low_memory"] = False
    check_file(path_obj, label)
    try:
        df = pd.read_csv(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

def safe_read_excel(path_obj, label, **kwargs):
    check_file(path_obj, label)
    try:
        df = pd.read_excel(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

# text columns, incl. the new pandas 3 "str" dtype (select_dtypes(["object", "string"]) misses it)
def text_columns(df):
    return [c for c in df.columns if pd.api.types.is_string_dtype(df[c].dtype)]

# Strip leading/trailing whitespace from selected string columns
def strip_whitespace_columns(df, columns, dataset_name="dataset"):
    existing_cols = [c for c in columns if c in df.columns]
    if not existing_cols:
        logging.info(f" - No whitespace-cleanup columns found in {dataset_name}")
        return df
    for col in existing_cols:
        # only process non-null values
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    logging.info(
        f" - Whitespace cleanup applied to {dataset_name} | "
        f"Columns: {', '.join(map(str, existing_cols[:20]))}{' ...' if len(existing_cols) > 20 else ''}" # max 20 names logged
    )
    return df

# - Strip leading/trailing spaces from column names (e.g., "InChIKey " → "InChIKey")
# - Done first, so that next step (values) also finds columns whose names had spaces (e.g., "InChIKey ")
# - Stops with an error if stripping would create two columns with the same name (e.g., "Name" and "Name ")
def strip_column_names(df, dataset_name="dataset"):
    old_names = list(df.columns)
    new_names = [c.strip() if isinstance(c, str) else c for c in old_names]
    changed = [o for o, n in zip(old_names, new_names) if o != n]

    if not changed:
        logging.info(f" - Column names already clean in {dataset_name}")
        return df

    duplicates = sorted({n for n, k in Counter(new_names).items() if k > 1}, key=str)
    if duplicates:
        raise ValueError(f"Stripping column names in {dataset_name} would create duplicate columns: {duplicates}")

    df.columns = new_names
    logging.info(f" - Column names stripped in {dataset_name}: {len(changed):,} | e.g. {[repr(c) for c in changed[:5]]}")
    return df


# Endpoint modeling plan & scaling helpers
def plan_endpoint(y, censor_frac=0.35, min_class_n=15, skew_log=1.0):
    """Suggests how to model one tox endpoint.
    - Many samples share the lowest value (e.g. <LOQ replaced by LOQ/sqrt(2), or 0 = no effect)
      → 'classification': active (above that value) vs. inactive.
    - Too few active OR inactive samples for that → 'skip'.
    - Otherwise 'regression', on log10 scale if strongly right-skewed."""
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna()
    if len(y) == 0:
        return dict(task="skip", transform="none", n=0, n_unique=0, mode_value=np.nan, n_mode=0,
                    frac_mode=np.nan, mode_is_min=False, n_active=0, skewness=np.nan, reason="no values")
    counts = y.value_counts()
    mode_val, n_mode = counts.index[0], int(counts.iloc[0])
    frac_mode = n_mode / len(y)
    mode_is_min = bool(np.isclose(mode_val, y.min()))
    n_active = int((~np.isclose(y, y.min())).sum())
    sk = float(skew(y)) if y.nunique() > 2 else np.nan

    if frac_mode >= censor_frac and mode_is_min:
        n_minor = min(n_active, len(y) - n_active)
        if n_minor >= min_class_n:
            task, reason = "classification", f"{frac_mode:.0%} of samples at lowest value"
        else:
            task, reason = "skip", f"only {n_minor} samples in the smaller class (< {min_class_n})"
        transform = "binary"
    else:
        task, reason = "regression", "continuous"
        transform = "log10" if (sk > skew_log and (y >= 0).all()) else "none"

    return dict(task=task, transform=transform, n=len(y), n_unique=int(y.nunique()), mode_value=mode_val,
                n_mode=n_mode, frac_mode=frac_mode, mode_is_min=mode_is_min, n_active=n_active,
                skewness=sk, reason=reason)

def r_scale(df):
    """Z-score like R's scale(): (x - mean) / sample SD (n - 1). Drops columns with SD = 0 or undefined."""
    sd = df.std(ddof=1)
    keep = sd.index[(sd > 0) & sd.notna()]
    return (df[keep] - df[keep].mean()) / sd[keep], sd.index.difference(keep)

# Seed
GLOBAL_SEED = 1
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

majorsep()
logging.info(f"Imports & logging setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Check if necessary columns exist in raw input data files
majorsep()
logging.info(f"Checking if necessary columns exist in raw input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Data folder
DATA_DIR = Path("../data") # working folder (data/ next to notebooks/); or the full path, e.g. Path(r"D:/projects/INQUIRE_TIP/data")

# input file names in one place
TOX_DUST_FILE = DATA_DIR / "tox_vs_dust_merged.csv" # merged tox vs. dust dataset
META_VU_FILE = DATA_DIR / "feature_metadata_LC-MS.xlsx" # feature metadata, LC-MS features
META_UMU_FILE = DATA_DIR / "feature_metadata_GCxGC-MS.xlsx" # feature metadata, GCxGC-MS features
META_CLASS_FILE = DATA_DIR / "feature_metadata_classified.xlsx" # feature metadata of all features + chemical class

# Output folder
OUT_DIR = Path(DATA_DIR / "inquire_tip_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True) # also creates missing parent folders

META_COLS = ["ML_feature_id", "rt1_(min)", "rt2_(sec)", "feature_id", "feature_name", "formula",
             "inchikey", "n_detected_samples", "detection_frequency", "concentration_type"]

# Merged tox vs. dust dataset
check_cols_before_load(TOX_DUST_FILE,
    ["sample_base", "%.reduced.cell.viability.per.mg_BEAS2B",
     "%.expr.of.IL6_0.1mg/ml_BEAS2B", "%.expr.of.IL6_1mg/ml_BEAS2B",
     "%.expr.of.IL8_0.1mg/ml_BEAS2B", "%.expr.of.IL8_1mg/ml_BEAS2B", "androgen.activity.(ng/g)",
     "anti-androgen.activity.(µg/g)", "AhR.activity.(ng/g)", "TR.agonism.(ng/g)", "estrogen.activity.(pg/g)",
     "anti-estrogen.activity.(ng/g)"],
    "Merged tox vs. dust dataset")

# Feature metadata
check_cols_before_load(META_VU_FILE, META_COLS, "Feature metadata VU")
check_cols_before_load(META_UMU_FILE, META_COLS, "Feature metadata UMU")

minorsep()
logging.info(f"Done checking if necessary columns exist in raw input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load raw input data
majorsep()
logging.info(f"Loading raw input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Merged tox vs. dust dataset
tox_vs_dust_0 = safe_read_csv(TOX_DUST_FILE, "Merged tox vs. dust dataset", low_memory=False)
minorsep()

# Feature metadata UMU
feat_meta_umu_0 = safe_read_excel(META_UMU_FILE, "Feature metadata UMU", engine="openpyxl")
minorsep()

# Feature metadata VU
feat_meta_vu_0 = safe_read_excel(META_VU_FILE, "Feature metadata VU", engine="openpyxl")
minorsep()

# Merge feature metadata VU & UMU
feat_meta_0 = pd.concat(
    [feat_meta_umu_0,
     feat_meta_vu_0],
    ignore_index=True # renumber rows 0, 1, 2, ...
)

logging.info(f" - feat_meta_umu: {feat_meta_umu_0.shape[0]} rows \n - feat_meta_vu: {feat_meta_vu_0.shape[0]} rows") # labels were swapped
logging.info(f" - feat_meta: {feat_meta_0.shape[0]} rows, {feat_meta_0.shape[1]} cols")
minorsep()

# Check that each ML_feature_id entry appears only once
n_dup_mlids = feat_meta_0["ML_feature_id"].duplicated().sum()
logging.info(f" - Duplicate ML_feature_id values: {n_dup_mlids}")
minorsep()

logging.info(f"Merged feature metadata:")
display(feat_meta_0)

minorsep()
logging.info(f"Done loading raw input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Whitespace cleanup
majorsep()
logging.info(f"Running whitespace cleanup for selected columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Stripping whitespace from column names")
for df_raw, df_label in [
    (tox_vs_dust_0, "Merged tox vs. dust dataset"),
    (feat_meta_0, "Feature metadata")
]:
    strip_column_names(df_raw, df_label)
minorsep()

# Strip whitespaces from merged tox vs. dust dataset column values
# Use all text columns in df_0 instead of a fixed list
logging.info(f"Stripping whitespace from merged tox vs. dust dataset column values")
strip_cols_all = text_columns(tox_vs_dust_0) # also catches the pandas 3 "str" dtype
logging.info(f" - {len(strip_cols_all)} of {tox_vs_dust_0.shape[1]} columns selected for whitespace cleanup")

# Report settings
MAX_SHOW_PER_COL = None  # max changed values printed per column; None = print all

df_before = tox_vs_dust_0[strip_cols_all].copy()  # snapshot before cleanup, for the change report

tox_vs_dust_strip = strip_whitespace_columns(tox_vs_dust_0, strip_cols_all, "Merged tox vs. dust dataset")

# Report what was stripped: column name, row index, old and new value
changed = df_before.notna() & df_before.ne(tox_vs_dust_strip[strip_cols_all])  # True where a value changed
changed_cols = changed.columns[changed.any()].tolist()
logging.info(f" - Columns with stripped values: {len(changed_cols)} | Values changed: {int(changed.values.sum())}")

for col in changed_cols:
    idx_changed = changed.index[changed[col]]
    logging.info(f" - stripped col {col}: {len(idx_changed)} values")
    for idx in idx_changed[:MAX_SHOW_PER_COL]:
        logging.info(f"     idx {idx}: {df_before.at[idx, col]!r} → {tox_vs_dust_strip.at[idx, col]!r}")
    if MAX_SHOW_PER_COL is not None and len(idx_changed) > MAX_SHOW_PER_COL:
        logging.info(f"     ... {len(idx_changed) - MAX_SHOW_PER_COL} more not shown")

del df_before, changed
minorsep()

# Strip whitespaces from feature metadata column values
# Use all text columns in df_0 instead of a fixed list
logging.info(f"Stripping whitespace from feature metadata column values")
strip_cols_all_meta = text_columns(feat_meta_0)
logging.info(f" - {len(strip_cols_all_meta)} of {feat_meta_0.shape[1]} columns selected for whitespace cleanup")

# Report settings
MAX_SHOW_PER_COL_META = None  # max changed values printed per column; None = print all

df_before_meta = feat_meta_0[strip_cols_all_meta].copy()  # snapshot before cleanup, for the change report

feat_meta_strip = strip_whitespace_columns(feat_meta_0, strip_cols_all_meta, "Feature metadata")

# Report what was stripped: column name, row index, old and new value
changed_meta = df_before_meta.notna() & df_before_meta.ne(feat_meta_strip[strip_cols_all_meta])  # True where a value changed
changed_cols_meta = changed_meta.columns[changed_meta.any()].tolist()
logging.info(f" - Columns with stripped values: {len(changed_cols_meta)} | Values changed: {int(changed_meta.values.sum())}")

for col in changed_cols_meta:
    idx_changed_meta = changed_meta.index[changed_meta[col]]
    logging.info(f" - stripped col {col}: {len(idx_changed_meta)} values")
    for idx in idx_changed_meta[:MAX_SHOW_PER_COL_META]:
        logging.info(f"     idx {idx}: {df_before_meta.at[idx, col]!r} → {feat_meta_strip.at[idx, col]!r}")
    if MAX_SHOW_PER_COL_META is not None and len(idx_changed_meta) > MAX_SHOW_PER_COL_META:
        logging.info(f"     ... {len(idx_changed_meta) - MAX_SHOW_PER_COL_META} more not shown")

del df_before_meta, changed_meta

minorsep()
logging.info(f"Done running whitespace cleanup for selected columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Harmonize column names
majorsep()
logging.info(f"Harmonizing column names | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_cols = tox_vs_dust_strip.copy()
feat_meta_cols = feat_meta_strip.copy()

logging.info(f"Merged tox vs. dust dataset cols [before renaming] ({len(tox_vs_dust_cols.columns)}; 15 cols printed):")
logging.info(f"{tox_vs_dust_cols.columns.tolist()[:15]}")
minorsep()

# Rename columns to match previous tox endpoint col name convention
TOX_COL_RENAME = {
    "sample_base": "Sample",
    "%.reduced.cell.viability.per.mg_BEAS2B": "reduced cell viability/mg BEAS-2B (effect cut-off = 20%)",
    "%.expr.of.IL6_0.1mg/ml_BEAS2B": "relative IL-6 expression (BEAS-2B 0.1 mg/ml)",
    "%.expr.of.IL6_1mg/ml_BEAS2B": "relative IL-6 expression (BEAS-2B 1 mg/ml)",
    "%.expr.of.IL8_0.1mg/ml_BEAS2B": "relative IL-8 expression (BEAS-2B 0.1 mg/ml)",
    "%.expr.of.IL8_1mg/ml_BEAS2B": "relative IL-8 expression (BEAS-2B 1 mg/ml)",
    "androgen.activity.(ng/g)": "andro (ng/g)",
    "anti-androgen.activity.(µg/g)": "antiandro (µg/g)",
    "AhR.activity.(ng/g)": "AhR (ng/g)",
    "TR.agonism.(ng/g)": "TR agonism (ng/g)",
    "estrogen.activity.(pg/g)": "estro (pg/g)",
    "anti-estrogen.activity.(ng/g)": "antiestro (ng/g)",
}

missing_cols = [c for c in TOX_COL_RENAME if c not in tox_vs_dust_cols.columns]
if missing_cols:
    logging.warning(f"Col not found, not renamed: {missing_cols}")
    minorsep()

tox_vs_dust_cols = tox_vs_dust_cols.rename(columns=TOX_COL_RENAME)
logging.info(f"Renamed {len(TOX_COL_RENAME) - len(missing_cols)} cols (15 cols printed): \n{tox_vs_dust_cols.columns.tolist()[:15]}")
minorsep()

# Harmonize column names
tox_vs_dust_cols.columns = (
    tox_vs_dust_cols.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Stop if harmonizing created duplicate column names
dup_cols = tox_vs_dust_cols.columns[tox_vs_dust_cols.columns.duplicated()].tolist()
if dup_cols:
    raise ValueError(f"Duplicate column names after harmonizing: {dup_cols[:20]}")

logging.info(f"Merged tox vs. dust dataset cols [after renaming] ({len(tox_vs_dust_cols.columns)}; 15 cols printed):")
logging.info(f"{tox_vs_dust_cols.columns.tolist()[:15]}")
minorsep()

# Harmonize column names - Feature metadata
feat_meta_cols.columns = (
    feat_meta_cols.columns
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

logging.info(f"Feature metadata cols ({len(feat_meta_cols.columns)} cols):")
logging.info(f"{feat_meta_cols.columns.tolist()}")
minorsep()

logging.info(f"Done harmonizing column names | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Drop "_<number>" suffixes from InChIKeys (if any)
majorsep()
logging.info(f"Dropping '_<number>' suffixes from InChIKeys (if any) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Runs only if the column exists; works when the column is empty or not text
if "inchikey" in feat_meta_cols.columns:
    inchikey_before = feat_meta_cols["inchikey"].copy()
    has_key = inchikey_before.notna()
    feat_meta_cols["inchikey"] = feat_meta_cols["inchikey"].astype(object)
    feat_meta_cols.loc[has_key, "inchikey"] = (
        inchikey_before[has_key].astype(str).str.replace(r"_[\d.]+$", "", regex=True)
    )
    inchikey_changed = has_key & (inchikey_before.astype(str) != feat_meta_cols["inchikey"].astype(str))
    logging.info(f"InChIKeys with suffix removed: {int(inchikey_changed.sum())}")
    minorsep()

    if inchikey_changed.any():
        logging.info(f"InChIKeys before and after:")
        show_cols = [c for c in ["ML_feature_id", "inchikey"] if c in feat_meta_cols.columns]
        display(pd.concat([inchikey_before[inchikey_changed].rename("inchikey_before"),
                           feat_meta_cols.loc[inchikey_changed, show_cols]], axis=1))
        minorsep()
else:
    logging.warning(f" - No 'inchikey' column in feature metadata; step skipped")
    minorsep()

# Save merged and cleaned feature metadata
feat_meta_cols.to_csv(OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_UMU_&_VU_feature_metadata_clean.csv", index=False)
logging.info(f"Saved cleaned feature metadata to: \n{OUT_DIR}")

minorsep()
logging.info(f"Dropped '_<number>' suffixes from InChIKeys (if any) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Define tox endpoint columns in merged tox vs. dust dataset
majorsep()
logging.info(f"Defining toxicity endpoint columns in merged tox vs. dust dataset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_work = tox_vs_dust_cols.copy()

SMPL_COL_MRGD = "Sample" # sample code col
LAST_TOX_COL = "antiestro (ng/g)" # last tox endpoint col
FIRST_FEAT_COL = "2427_0.69_UMU" # first feature col

cols = list(tox_vs_dust_work.columns)

# clear error if a key column is missing
for key_col in [SMPL_COL_MRGD, FIRST_FEAT_COL]:
    if key_col not in cols:
        raise ValueError(f"Column '{key_col}' not found in merged tox vs. dust dataset")
if cols.index(SMPL_COL_MRGD) >= cols.index(FIRST_FEAT_COL):
    raise ValueError(f"'{SMPL_COL_MRGD}' must come before '{FIRST_FEAT_COL}'")

start_idx = cols.index(SMPL_COL_MRGD) + 1 # first col after SAMPLE_COL
end_idx = cols.index(FIRST_FEAT_COL) # position of FIRST_FEAT_COL (excluded)
TOX_ENDPTS_MRGD = cols[start_idx:end_idx]
DUST_FEAT_MRGD = cols[end_idx:] # moved here from the dust diagnostics cell

# Sanity checks on the endpoint block
if len(TOX_ENDPTS_MRGD) == 0:
    raise ValueError("No tox endpoint columns found between Sample and the first feature column")
if TOX_ENDPTS_MRGD[-1] != LAST_TOX_COL:
    logging.warning(f" - Last tox endpoint column is '{TOX_ENDPTS_MRGD[-1]}', expected '{LAST_TOX_COL}'")

logging.info(f"Sample column (1 col): \n - {SMPL_COL_MRGD}")
minorsep()
logging.info(f"Tox endpoint columns ({len(TOX_ENDPTS_MRGD)} cols):")
for col in TOX_ENDPTS_MRGD:
    logging.info(f" - {col}")
minorsep()
logging.info(f"Dust feature columns: {len(DUST_FEAT_MRGD):,}")

minorsep()
logging.info(f"Done defining toxicity endpoint columns in merged tox vs. dust dataset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Tox sample QC for merged tox vs. dust dataset
majorsep()
logging.info(f"Tox sample QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_smpl_qc = tox_vs_dust_work.copy()
n_smpls_mrgd = len(tox_vs_dust_smpl_qc)
n_uniq_smpls_mrgd = (tox_vs_dust_smpl_qc[SMPL_COL_MRGD].nunique(dropna=True))

logging.info(f" - Total rows: {n_smpls_mrgd:,}")
logging.info(f" - Unique samples: {n_uniq_smpls_mrgd:,}")

dup_smpls_mrgd = (tox_vs_dust_smpl_qc[SMPL_COL_MRGD].duplicated().sum())
n_missing_smpl_ids = tox_vs_dust_smpl_qc[SMPL_COL_MRGD].isna().sum()

logging.info(f" - Duplicate sample IDs: {dup_smpls_mrgd:,}")
logging.info(f" - Rows without sample ID: {n_missing_smpl_ids:,}")

if dup_smpls_mrgd > 0:
    logging.warning(f" - Duplicate sample IDs detected (only the first row of each is used later): "
                    f"{tox_vs_dust_smpl_qc.loc[tox_vs_dust_smpl_qc[SMPL_COL_MRGD].duplicated(), SMPL_COL_MRGD].tolist()[:20]}")

if n_missing_smpl_ids > 0:
    logging.warning(f" - Rows without sample ID are dropped later")

minorsep()
logging.info(f"Done tox sample QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Raw tox content QC
majorsep()
logging.info(f"Raw tox content QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_raw_qc = tox_vs_dust_smpl_qc.copy()
tox_vs_dust_qc_report = []

for endpoint in TOX_ENDPTS_MRGD:
    # NaNs counted BEFORE converting to text (after astype(str) they became "nan" and were not counted);
    # "whitespace" check removed (always 0 after strip); "non_numeric" added
    ser_raw = tox_vs_dust_raw_qc[endpoint]
    ser_txt = ser_raw.astype(str).str.strip()
    ser_num = pd.to_numeric(ser_raw, errors="coerce")
    n_nan_tox_vs_dust = int(ser_raw.isna().sum())
    n_lod_tox_vs_dust = int(ser_txt.str.startswith("<").sum())
    n_neg_tox_vs_dust = int((ser_num < 0).sum())
    n_zero_tox_vs_dust = int((ser_num == 0).sum())
    n_empty_tox_vs_dust = int((ser_raw.notna() & ser_txt.eq("")).sum())
    n_nt_tox_vs_dust = int(ser_txt.str.lower().isin(["nt", "n.t.", "not tested"]).sum())
    n_text_tox_vs_dust = int((ser_raw.notna() & ser_num.isna()).sum())

    tox_vs_dust_qc_report.append([
        endpoint,
        n_nan_tox_vs_dust,
        n_lod_tox_vs_dust,
        n_neg_tox_vs_dust,
        n_zero_tox_vs_dust,
        n_empty_tox_vs_dust,
        # n_space_tox_vs_dust,
        n_nt_tox_vs_dust,
        n_text_tox_vs_dust
    ])

tox_vs_dust_qc_report = pd.DataFrame(
    tox_vs_dust_qc_report,
    columns=[
        "endpoint",
        "nan",
        "<lod",
        "negative",
        "zero",
        "empty_cell",
        "nt_value",
        "non_numeric" # text entries; become NaN in the next cell
    ]
)

display(tox_vs_dust_qc_report)

minorsep()
logging.info(f"Done raw tox content QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Tox QC summary (merged tox vs. dust dataset)
majorsep()
logging.info(f"Tox QC summary (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_clean = tox_vs_dust_raw_qc.copy()

# Make tox endpoints numeric (text entries → NaN), so comparisons and models don't fail
for endpoint in TOX_ENDPTS_MRGD:
    num = pd.to_numeric(tox_vs_dust_clean[endpoint], errors="coerce")
    n_coerced = int((tox_vs_dust_clean[endpoint].notna() & num.isna()).sum())
    if n_coerced > 0:
        logging.warning(f" - {endpoint}: {n_coerced} non-numeric entries set to NaN")
    tox_vs_dust_clean[endpoint] = num

# tox_vs_dust_final_qc = tox_vs_dust_clean.copy()
tox_vs_dust_final_qc = tox_vs_dust_clean # no extra copy needed
tox_vs_dust_final_report = []

for endpoint in TOX_ENDPTS_MRGD:
    tox_vs_dust_final_report.append([
        endpoint,
        tox_vs_dust_final_qc[endpoint].notna().sum(),
        tox_vs_dust_final_qc[endpoint].isna().sum(),
        round((tox_vs_dust_final_qc[endpoint].isna().sum() * 100 / len(tox_vs_dust_final_qc)), 1),
        (tox_vs_dust_final_qc[endpoint] < 0).sum(),
        round(tox_vs_dust_final_qc[endpoint].median(skipna=True), 1),
        round(tox_vs_dust_final_qc[endpoint].mean(skipna=True), 1),
    ])

tox_vs_dust_final_report = pd.DataFrame(
    tox_vs_dust_final_report,
    columns=[
        "endpoint",
        "non_missing",
        "missing",
        "missing %",
        "negative",
        "median",
        "mean"
    ]
)

display(tox_vs_dust_final_report)

minorsep()
logging.info(f"tox_vs_dust_clean shape [samples, columns]: {tox_vs_dust_clean.shape}")

minorsep()
logging.info(f"Done tox QC summary (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create tox matrix
majorsep()
logging.info(f"Tox matrix diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

tox_vs_dust_mtrx = tox_vs_dust_final_qc.copy()
tox_mtrx = tox_vs_dust_mtrx[[SMPL_COL_MRGD] + TOX_ENDPTS_MRGD].copy() # by column name, not position
logging.info(f"tox_mtrx shape: {tox_mtrx.shape}") # rows, 1 + n endpoints
minorsep()
logging.info(list(tox_mtrx.columns))
minorsep()

# Endpoint distribution summary
logging.info(f"Endpoint distribution summary:")

endpoint_summary_mrgd = []

for endpoint in [c for c in tox_mtrx.columns if c != SMPL_COL_MRGD]: # skip "Sample" col
    s = pd.to_numeric(tox_mtrx[endpoint], errors="coerce")  # text/odd entries → NaN
    s = s.dropna() # drop NaNs for this col only

    endpoint_summary_mrgd.append([
        endpoint,
        s.count(),
        tox_mtrx[endpoint].isna().sum(), # NaNs in the original col
        s.nunique(dropna=True),
        s.min(),
        s.quantile(0.01),
        s.quantile(0.05),
        s.median(),
        s.quantile(0.95),
        s.quantile(0.99),
        s.max()
    ])

endpoint_summary_df = pd.DataFrame(
    endpoint_summary_mrgd,
    columns=[
        "endpoint",
        "n_non_missing",
        "n_missing",
        "n_unique",
        "min",
        "p01",
        "p05",
        "median",
        "p95",
        "p99",
        "max"]
)

display(endpoint_summary_df.round(3))

# Skewness
minorsep()
logging.info(f"Skewness of tox matrix:")

skew_rows = []
for endpoint in TOX_ENDPTS_MRGD:
    s = pd.to_numeric(tox_mtrx[endpoint], errors="coerce").dropna() # text → NaN, then drop NaNs
    skew_rows.append([endpoint, s.count(), round(float(skew(s)), 2) if s.nunique() > 2 else np.nan]) # NaN instead of warning for (near-)constant endpoints

skew_df = pd.DataFrame(skew_rows, columns=["endpoint", "n_valid", "skewness"])

with pd.option_context("display.max_colwidth", None):
    display(skew_df)

minorsep()
logging.info(f"Done creating tox matrix diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create sample list
majorsep()
logging.info(f"Creating sample list (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# SMPL_COL_MRGD
mrgd_names = set(tox_vs_dust_mtrx[SMPL_COL_MRGD].dropna()) # no NaN sample IDs; unused duplicate set removed

SAMPLES_MRGD = sorted(mrgd_names, key=str) # works with mixed ID types

logging.info(f" - Samples (SMPL_COL_MRGD): {len(SAMPLES_MRGD):,}")

minorsep()
logging.info(f"Done creating sample list (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create dust predictor matrix (from merged tox vs. dust dataset)
majorsep()
logging.info(f"Creating dust predictor matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

dust_mtrx = tox_vs_dust_mtrx[[SMPL_COL_MRGD] + DUST_FEAT_MRGD].copy() # by column name, not position

# Quick checks
logging.info(f"tox_mtrx shape: {tox_mtrx.shape}, \ndust_mtrx shape: {dust_mtrx.shape}")
minorsep()
logging.info(f"tox_mtrx cols: \n{list(tox_mtrx.columns)}")
minorsep()
logging.info(f"dust_mtrx cols (first 10): \n{list(dust_mtrx.columns[:10])}")
minorsep()

# drop rows without sample ID and keep only the first row of duplicated sample IDs
tox_mtrx = tox_mtrx.dropna(subset=[SMPL_COL_MRGD]).drop_duplicates(subset=SMPL_COL_MRGD, keep="first")
dust_mtrx = dust_mtrx.dropna(subset=[SMPL_COL_MRGD]).drop_duplicates(subset=SMPL_COL_MRGD, keep="first")

# Make "Sample" the index
tox_mtrx_smpl_idx = tox_mtrx.set_index(SMPL_COL_MRGD)
dust_mtrx_smpl_idx = dust_mtrx.set_index(SMPL_COL_MRGD)

# Quick checks
logging.info(f"tox_mtrx_smpl_idx shape: {tox_mtrx_smpl_idx.shape}, \ndust_mtrx_smpl_idx shape: {dust_mtrx_smpl_idx.shape}")
minorsep()
logging.info(f"Duplicate sample codes in tox_mtrx_smpl_idx index: {tox_mtrx_smpl_idx.index.duplicated().sum()}")
logging.info(f"Duplicate sample codes in dust_mtrx_smpl_idx index: {dust_mtrx_smpl_idx.index.duplicated().sum()}")
minorsep()
logging.info(f"tox_mtrx_smpl_idx cols: \n{list(tox_mtrx_smpl_idx.columns)}")
minorsep()
logging.info(f"dust_mtrx_smpl_idx cols (first 10): \n{list(dust_mtrx_smpl_idx.columns[:10])}")
minorsep()

X_tox_vs_dust = dust_mtrx_smpl_idx.copy()
X_tox_vs_dust = (X_tox_vs_dust.loc[SAMPLES_MRGD])

# make every dust column numeric (text entries → NaN); only non-numeric columns are converted
non_num_feat = [c for c in X_tox_vs_dust.columns if not pd.api.types.is_numeric_dtype(X_tox_vs_dust[c])]
if non_num_feat:
    n_na_before = X_tox_vs_dust[non_num_feat].isna().sum().sum()
    X_tox_vs_dust[non_num_feat] = X_tox_vs_dust[non_num_feat].apply(pd.to_numeric, errors="coerce")
    logging.warning(f" - {len(non_num_feat)} non-numeric dust columns converted; "
                    f"{X_tox_vs_dust[non_num_feat].isna().sum().sum() - n_na_before} text entries set to NaN | e.g. {non_num_feat[:5]}")
    minorsep()

logging.info(f"X_tox_vs_dust shape [samples, features]: {X_tox_vs_dust.shape}")

minorsep()
logging.info(f"Done creating dust predictor matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Dust matrix diagnostics
# - Computed for all ~27000 features at once
# - Adds number and fraction of samples with a detection (> 0)
majorsep()
logging.info(f"Dust matrix diagnostics | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

dust_num = X_tox_vs_dust # numeric, samples x features (raw, unscaled)

# Feature distribution summary
logging.info(f"Feature distribution summary:")

dust_q = dust_num.quantile([0.01, 0.05, 0.5, 0.95, 0.99]).T
feat_summary_df = pd.DataFrame({
    "feature": dust_num.columns,
    "n_non_missing": dust_num.count().values,
    "n_missing": dust_num.isna().sum().values,
    "n_unique": dust_num.nunique().values,
    "n_detected": (dust_num > 0).sum().values,
    "detection_frac": (dust_num > 0).mean().values,
    "n_negative": (dust_num < 0).sum().values,
    "min": dust_num.min().values,
    "p01": dust_q[0.01].values,
    "p05": dust_q[0.05].values,
    "median": dust_q[0.5].values,
    "p95": dust_q[0.95].values,
    "p99": dust_q[0.99].values,
    "max": dust_num.max().values,
})

display(feat_summary_df.round(3))

# Skewness
minorsep()
logging.info(f"Skewness of dust matrix:")

with warnings.catch_warnings(), np.errstate(all="ignore"):
    warnings.simplefilter("ignore") # constant features give "precision loss" warnings (skewness = NaN)
    skew_vals_dust = np.ma.filled(skew(dust_num.values, axis=0, nan_policy="omit"), np.nan)

skew_df_dust = pd.DataFrame({
    "feature": dust_num.columns,
    "n_valid": dust_num.count().values,
    "skewness": np.round(np.asarray(skew_vals_dust, dtype=float), 2),
})

with pd.option_context("display.max_colwidth", None):
    display(skew_df_dust)

# SD = 0 (constant) features
minorsep()
sd_vals_dust = dust_num.std(ddof=1) # sample SD (n - 1), NaNs skipped, as in R
ZERO_SD_FEATS = sd_vals_dust[sd_vals_dust == 0].index

logging.info(f"Constant features (SD = 0): {len(ZERO_SD_FEATS)} out of {len(sd_vals_dust)}")
minorsep()

# Distribution summary of SD = 0 features
logging.info(f"Feature distribution summary of SD = 0 features:")
feat_summary_zero_sd_df = feat_summary_df[feat_summary_df["feature"].isin(ZERO_SD_FEATS)]
display(feat_summary_zero_sd_df.round(3))

# Detection frequency overview (used by the dust preprocessing cell below)
minorsep()
logging.info(f"Features by detection frequency (fraction of samples with value > 0):")
display(pd.cut(feat_summary_df["detection_frac"], [-0.01, 0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0])
        .value_counts().sort_index().rename("n_features").to_frame())

minorsep()
logging.info(f"Dust matrix diagnostics done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Detection-frequency threshold check (run before the dust preprocessing cell)
# Shows how many features each MIN_DETECT_FRAC would keep, per lab, to choose the threshold
majorsep()
logging.info(f"Detection-frequency threshold check | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

dust_check = X_tox_vs_dust.fillna(0)
det_frac_check = (dust_check > 0).mean() # negatives (below LOD) count as not detected
n_uniq_check = dust_check.abs().nunique()
lab_check = (pd.Series(dust_check.columns, index=dust_check.columns).astype(str)
             .str.extract(r"_(UMU|VU)(?=_|$)")[0].fillna("other")) # lab tag followed by any suffix or the end

threshold_rows = []
for thr in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]:
    kept = (det_frac_check >= thr) & (n_uniq_check >= 5)
    row = {"MIN_DETECT_FRAC": thr, "min_n_samples": int(np.ceil(thr * len(dust_check)))}

    for lab in sorted(lab_check.unique()): # "other" column only appears if such IDs exist
        row[f"{lab}_kept"] = int((kept & (lab_check == lab)).sum())
    row["total_kept"] = int(kept.sum())
    threshold_rows.append(row)

threshold_check_df = pd.DataFrame(threshold_rows)
lab_totals = lab_check.value_counts()
logging.info(f"Features per lab: {lab_totals.to_dict()}")
minorsep()
display(threshold_check_df)
minorsep()

# Detection-frequency distribution per lab
logging.info(f"Number of features by detection frequency, per lab:")
det_bins = pd.cut(det_frac_check, [-0.01, 0, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0])
display(pd.crosstab(det_bins, lab_check))
minorsep()

# Features whose ID contains neither _UMU nor _VU
other_feats = lab_check.index[lab_check == "other"]
logging.info(f"Features without an _UMU or _VU lab tag: {len(other_feats)}")
for f in other_feats:
    logging.info(f" - {f!r} | detected in {det_frac_check[f]:.0%} of samples")

minorsep()
logging.info(f"Done detection-frequency threshold check | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Dust feature preprocessing (new cell)
# - NaN -> 0 (not detected)
# - Detection-frequency filter, computed BEFORE the sign handling: detected = value > 0;
#   negative values (= below LOD, marked with a minus sign; GCxGC-MS only) count as NOT detected
#   Keep features detected in >= MIN_DETECT_FRAC of samples and with >= MIN_UNIQUE_VALUES distinct values
# - Negative values: NEG_HANDLING = "abs" (default) -> |value|, i.e. keep the below-LOD concentration
#   estimate; "zero" -> 0 (substitution, old approach)
# - log10(x + c), c = half of the smallest DETECTED (originally > 0) value per feature, so below-LOD
#   estimates stay compressed near the floor and their low-precision differences can't dominate
majorsep()
logging.info(f"Dust feature preprocessing | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

NAN_TO_ZERO = True # NaN in dust data = not detected
# NEG_TO_ZERO = True # negative concentrations = not detected
NEG_HANDLING = "abs" # "abs" = keep below-LOD estimates as |value|; "zero" = set to 0
# MIN_DETECT_FRAC = 0.10 # keep features detected in >= 20% of samples (~33 of 167); try 0.10-0.30
MIN_DETECT_FRAC = 0.10 # keep features detected in >= 10% of samples (e.g., 17 of 167); try 0.10-0.30
MIN_UNIQUE_VALUES = 5 # keep features with >= 5 distinct values
LOG_TRANSFORM_X = True

if NEG_HANDLING not in ["abs", "zero"]:
    raise ValueError("NEG_HANDLING must be 'abs' or 'zero'")

X_tox_vs_dust_prep = X_tox_vs_dust.copy()

n_nan_dust = int(X_tox_vs_dust_prep.isna().sum().sum())
logging.info(f" - NaNs in dust data: {n_nan_dust:,}")
if n_nan_dust > 0 and NAN_TO_ZERO:
    X_tox_vs_dust_prep = X_tox_vs_dust_prep.fillna(0)
    logging.info(f"   -> set to 0 (not detected)")

# Detection mask before sign handling (negative = below LOD = not detected)
det_mask = X_tox_vs_dust_prep > 0
neg_mask = X_tox_vs_dust_prep < 0

n_neg_dust = int(neg_mask.sum().sum())
n_feat_neg = int(neg_mask.any().sum())
# logging.info(f" - Negative values in dust data: {n_neg_dust:,} (in {n_feat_neg:,} features)")
logging.info(f" - Negative values (below LOD) in dust data: {n_neg_dust:,} (in {n_feat_neg:,} features)")
minorsep()

# Detection-frequency filter
# det_frac = (X_tox_vs_dust_prep > 0).mean()
det_frac = det_mask.mean() # negatives count as not detected
n_uniq = X_tox_vs_dust_prep.abs().nunique() # distinct values after sign handling
keep_feat = (det_frac >= MIN_DETECT_FRAC) & (n_uniq >= MIN_UNIQUE_VALUES)

# feat_lab = pd.Series(X_tox_vs_dust_prep.columns, index=X_tox_vs_dust_prep.columns).str.extract(r"_(UMU|VU)(?:_\d+)?$")[0].fillna("other")
feat_lab = (pd.Series(X_tox_vs_dust_prep.columns, index=X_tox_vs_dust_prep.columns).astype(str)
            .str.extract(r"_(UMU|VU)(?=_|$)")[0].fillna("other")) # lab tag followed by any suffix or the end

# warn if any ID still has no lab tag
n_other = int((feat_lab == "other").sum())
if n_other > 0:
    logging.warning(f" - {n_other} features without an _UMU or _VU lab tag: {feat_lab.index[feat_lab == 'other'].tolist()[:10]}")

filter_report = pd.DataFrame({
    "features_before": feat_lab.value_counts(),
    "features_kept": feat_lab[keep_feat].value_counts(),
}).fillna(0).astype(int)
filter_report.loc["total"] = filter_report.sum()
filter_report["kept_%"] = (100 * filter_report["features_kept"] / filter_report["features_before"]).round(1)

logging.info(f"Detection-frequency filter (detected in >= {MIN_DETECT_FRAC:.0%} of samples, >= {MIN_UNIQUE_VALUES} distinct values):")
display(filter_report)

X_tox_vs_dust_prep = X_tox_vs_dust_prep.loc[:, keep_feat]
det_mask = det_mask.loc[:, keep_feat]
neg_mask = neg_mask.loc[:, keep_feat]
if X_tox_vs_dust_prep.shape[1] == 0:
    raise ValueError("No dust features left after the detection-frequency filter; lower MIN_DETECT_FRAC")
minorsep()

# Sign handling of below-LOD values (kept features only)
logging.info(f" - Below-LOD values in kept features: {int(neg_mask.sum().sum()):,} (in {int(neg_mask.any().sum()):,} features)")
if NEG_HANDLING == "abs":
    X_tox_vs_dust_prep = X_tox_vs_dust_prep.abs()
    logging.info(f"   -> sign removed (below-LOD concentration estimates kept)")

    # QC: features where a below-LOD estimate exceeds a real detection in another sample
    max_below_lod = X_tox_vs_dust_prep.where(neg_mask).max()
    min_detected = X_tox_vs_dust_prep.where(det_mask).min()
    overlap = (max_below_lod > min_detected) & neg_mask.any()
    logging.info(f" - Features where a below-LOD value exceeds the smallest detection: \n   {int(overlap.sum()):,} "
                 f"of {int(neg_mask.any().sum()):,} (expected when LODs differ between samples)")
else:
    X_tox_vs_dust_prep = X_tox_vs_dust_prep.clip(lower=0)
    logging.info(f"   -> set to 0 (not detected)")
minorsep()

# log10 transform
if LOG_TRANSFORM_X:
    log_offset_x = X_tox_vs_dust_prep.where(det_mask).min() / 2 # from real detections only
    X_tox_vs_dust_prep = np.log10(X_tox_vs_dust_prep + log_offset_x)
    logging.info(f" - log10(x + c) applied; c = half of the smallest detected value per feature")

with warnings.catch_warnings(), np.errstate(all="ignore"):
    warnings.simplefilter("ignore")
    skew_after = pd.Series(np.ma.filled(skew(X_tox_vs_dust_prep.values, axis=0), np.nan))
logging.info(f" - Median feature skewness after preprocessing: {np.nanmedian(skew_after):.2f}")
logging.info(f" - X_tox_vs_dust_prep shape [samples, features]: {X_tox_vs_dust_prep.shape}")

# Save which features were kept, for the methods section
pd.DataFrame({"ML_feature_id": det_frac.index, "detection_frac": det_frac.values,
              "n_unique": n_uniq.values, "kept": keep_feat.values}).to_csv(
    OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_dust_feature_filter.csv", index=False)

minorsep()
logging.info(f"Done dust feature preprocessing | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Scale dust predictor matrix (from merged tox vs. dust dataset)
majorsep()
logging.info(f"Scaling dust predictor matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Scales preprocessed matrix (detection-filtered + log10); r_scale() = R scale() and drops SD = 0 columns
X_tox_vs_dust_scaled, cols_to_drop_X = r_scale(X_tox_vs_dust_prep)
logging.info(f" - Scaled {X_tox_vs_dust_prep.shape[1]:,} features (R scale() equivalent)")
logging.info(f" - Dropped constant/undefined-SD columns: {len(cols_to_drop_X)}; {X_tox_vs_dust_scaled.shape[1]:,} remain")
minorsep()
logging.info(f" - X_tox_vs_dust_scaled shape [samples, features]: {X_tox_vs_dust_scaled.shape}")
minorsep()

logging.info(f"X_tox_vs_dust_scaled:")
display(X_tox_vs_dust_scaled.head())

minorsep()
logging.info(f"Done scaling dust predictor matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create response matrix
majorsep()
logging.info(f"Creating response matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

Y_tox_vs_dust = tox_mtrx_smpl_idx.copy()
Y_tox_vs_dust = (Y_tox_vs_dust.loc[SAMPLES_MRGD])
Y_tox_vs_dust = Y_tox_vs_dust.apply(pd.to_numeric, errors="coerce") # numeric endpoints

logging.info(f" - Y_tox_vs_dust shape [samples, tox endpoints]: {Y_tox_vs_dust.shape}")
logging.info(f" - X_tox_vs_dust shape [samples, compounds]: {X_tox_vs_dust_scaled.shape}")

if len(X_tox_vs_dust_scaled) != len(Y_tox_vs_dust):
    raise ValueError(f" - X and Y have different row counts")

if not (X_tox_vs_dust_scaled.index.equals(Y_tox_vs_dust.index)):
    raise ValueError(f" - Sample order mismatch between X and Y")

minorsep()
logging.info(f" - Sample alignment verified: \n   X and Y have the same row counts and sample order")

minorsep()
logging.info(f"Done creating response matrix (from merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Final pre-ML tox endpoints QC (merged tox vs. dust dataset)
# - NaNs excluded from all counts and percentages
# - Adds the suggested modeling plan per endpoint (see plan_endpoint() in the imports cell):
#   - regression (log10 if skewed)
#   - classification (many samples at the lowest value)
#   - skip endpoint
majorsep()
logging.info(f"Final pre-ML tox endpoints QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

endpoint_info = []

for endpoint in TOX_ENDPTS_MRGD:
    p = plan_endpoint(Y_tox_vs_dust[endpoint])
    endpoint_info.append([
        endpoint,
        len(Y_tox_vs_dust),
        p["n"],
        len(Y_tox_vs_dust) - p["n"],
        p["n_unique"],
        p["mode_value"],
        p["n_mode"],
        round(100 * p["frac_mode"], 1) if p["n"] else np.nan,
        p["mode_is_min"],
        round(p["skewness"], 2) if pd.notna(p["skewness"]) else np.nan,
        p["task"],
        p["transform"],
        p["reason"]
    ])

endpoint_info = pd.DataFrame(
    endpoint_info,
    columns=[
        "endpoint",
        "n_total",
        "n_valid",
        "n_nan",
        "n_unique",
        "most_common_value",
        "n_most_common_value",
        "%_most_common_value",
        "most_common_is_min",
        "skewness",
        "suggested_task",
        "suggested_transform",
        "reason"
    ]
)

with pd.option_context("display.max_colwidth", None):
    display(endpoint_info)

endpoint_info.to_csv(OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_endpoint_modeling_plan.csv", index=False)

minorsep()
logging.info(f"Done final pre-ML tox endpoints QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Final pre-ML QC (merged tox vs. dust dataset)
majorsep()
logging.info(f"Final pre-ML QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f" - X shape [samples, features]: {X_tox_vs_dust_scaled.shape}")
logging.info(f" - Y shape [samples, tox endpoints]: {Y_tox_vs_dust.shape}")
minorsep()

if X_tox_vs_dust_scaled.isna().sum().sum() > 0:
    logging.warning(f"!!! NaNs found in X: {int(X_tox_vs_dust_scaled.isna().sum().sum())} (models will fail; check NAN_TO_ZERO)")
    minorsep()

if Y_tox_vs_dust.isna().sum().sum() > 0:
    logging.info(f"NaNs in Y: {int(Y_tox_vs_dust.isna().sum().sum())} (OK: those samples are dropped per endpoint at endpoint selection)")
    minorsep()

logging.info(f"Done final pre-ML QC (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save preprocessed datasets (merged tox vs. dust dataset)
majorsep()
logging.info(f"Saving preprocessed dust & tox endpoints datasets (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Remove empty "Sample" row if present
X_tox_vs_dust_scaled_row_rmv = X_tox_vs_dust_scaled.drop(index="Sample", errors="ignore")

# Read classified feature metadata file
check_cols_before_load(META_CLASS_FILE,
    ["ML_feature_id", "rt1_(min)", "rt2_(sec)", "feature_id", "feature_name", "formula",
     "inchikey", "class", "n_detected_samples", "detection_frequency", "concentration_type"],
    "Classified feature metadata")

# Classified feature metadata
feat_meta_class = safe_read_excel(META_CLASS_FILE, "Classified feature metadata", engine="openpyxl")
feat_meta_class.columns = [str(c).strip() for c in feat_meta_class.columns]
feat_meta_class["ML_feature_id"] = feat_meta_class["ML_feature_id"].astype(str).str.strip() # IDs as clean text, like the dust columns
minorsep()

# QC: duplicate ML_feature_id values in metadata
dup_mask = feat_meta_class["ML_feature_id"].duplicated(keep=False)
logging.info(f" - Duplicate ML_feature_id in metadata: \n   {feat_meta_class.loc[dup_mask, 'ML_feature_id'].nunique()} IDs, {dup_mask.sum()} rows")
minorsep()

if dup_mask.any():
    logging.info(f"All duplicate rows (sorted by ID):")
    display(feat_meta_class[dup_mask].sort_values("ML_feature_id"))
    minorsep()

    n_rows_before = len(feat_meta_class)
    feat_meta_class = feat_meta_class.drop_duplicates()  # remove exact copies (all cols identical)
    logging.info(f" - Exact duplicate rows removed: {n_rows_before - len(feat_meta_class)}")
    minorsep()

    conflict_mask = feat_meta_class["ML_feature_id"].duplicated(keep=False)
    if conflict_mask.any():
        logging.info(f" - WARNING: {feat_meta_class.loc[conflict_mask, 'ML_feature_id'].nunique()} IDs have duplicates with DIFFERENT info; keeping the first row of each:")
        display(feat_meta_class[conflict_mask].sort_values("ML_feature_id"))
        feat_meta_class = feat_meta_class.drop_duplicates(subset="ML_feature_id", keep="first")
        minorsep()

    feat_meta_class = feat_meta_class.reset_index(drop=True)
    logging.info(f" - feat_meta_class rows after cleanup: {len(feat_meta_class)}")
minorsep()

# Metadata → rows
meta_rows_mrgd = (feat_meta_class.set_index("ML_feature_id").T)

# Feature IDs present in both tables (keeps the column order of the dust matrix)
x_feats = X_tox_vs_dust_scaled_row_rmv.columns
common_feat_mrgd = x_feats[x_feats.isin(meta_rows_mrgd.columns)]

logging.info(f" - Features in dust matrix (after detection filter): {len(x_feats)}, in metadata: {len(meta_rows_mrgd.columns)}, in both: {len(common_feat_mrgd)}")
logging.info(f" - Dropped from metadata (not in dust matrix; mostly removed by the detection filter): {len(meta_rows_mrgd.columns) - len(common_feat_mrgd)}")
logging.info(f" - Dropped from dust matrix (no metadata): {len(x_feats) - len(common_feat_mrgd)}")
minorsep()

# QC: dust features with no metadata
missing_feats = x_feats[~x_feats.isin(meta_rows_mrgd.columns)]
if len(missing_feats) > 0:
    logging.info(f"Dust features with no metadata: {len(missing_feats)}")
    meta_ids = feat_meta_class["ML_feature_id"].astype(str)

    for f in missing_feats[:20]: # max 20 shown
        minorsep()
        logging.info(f"ID as stored: {f!r} (type: {type(f).__name__})")  # repr shows hidden spaces/characters

        # Same ID ignoring spaces and upper/lower case
        near = feat_meta_class[meta_ids.str.strip().str.lower() == str(f).strip().lower()]
        logging.info(f" - Metadata rows matching when ignoring spaces/case: {len(near)}")
        if len(near) > 0:
            display(near)

        # Same feature number and same lab suffix (e.g. "2427_..._UMU")
        parts = str(f).strip().split("_")
        similar = feat_meta_class[meta_ids.str.startswith(parts[0] + "_") & meta_ids.str.endswith("_" + parts[-1])]
        logging.info(f" - Metadata rows with same number '{parts[0]}' and lab '{parts[-1]}': {len(similar)}")
        if len(similar) > 0:
            display(similar.head(10))
        minorsep()

        # The feature's values in the dust data
        if f in X_tox_vs_dust.columns:
            logging.info(f"Unscaled dust values:")
            display(X_tox_vs_dust[f].describe().to_frame().T)
    minorsep()

# Cleaned metadata table: only features present in both tables
feat_meta_dust_masked = (
    feat_meta_class[feat_meta_class["ML_feature_id"].isin(common_feat_mrgd)]
    .reset_index(drop=True)
)
logging.info(f" - feat_meta_dust_masked shape: {feat_meta_dust_masked.shape}")
minorsep()
logging.info(f"feat_meta_dust_masked:")
display(feat_meta_dust_masked)
minorsep()

# Reindex both tables to the same feature set
meta_rows_mrgd = meta_rows_mrgd.reindex(columns=common_feat_mrgd)
X_tox_vs_dust_scaled_idx = X_tox_vs_dust_scaled_row_rmv.reindex(columns=common_feat_mrgd)

# Combine metadata rows + sample matrix
X_tox_vs_dust_meta = pd.concat([meta_rows_mrgd, X_tox_vs_dust_scaled_idx], axis=0)

logging.info(f"X_tox_vs_dust_meta (first 15 rows):")
display(X_tox_vs_dust_meta.head(15))
minorsep()

# Save
X_tox_vs_dust_meta.to_csv(OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_dust_detfilt_log10_scale().csv", index=True) # name says what was done
Y_tox_vs_dust.to_csv(OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_tox_endpoints_unscaled.csv", index=True)
feat_meta_dust_masked.to_csv(OUT_DIR / f"{datetime.now().strftime('%y%m%d')}_feature_metadata_dust_masked.csv", index=False)

# Files used by Pt2 (modeling)
run_date = datetime.now().strftime('%y%m%d')
X_tox_vs_dust_prep.to_csv(OUT_DIR / f"{run_date}_dust_detfilt_log10.csv", index=True) # modeling matrix: filtered + log10, not scaled (scaled per endpoint in Pt2)
X_tox_vs_dust.to_csv(OUT_DIR / f"{run_date}_dust_raw_numeric.csv", index=True) # raw numeric dust data, all features (univariate screen in Pt2)
feat_meta_class.to_csv(OUT_DIR / f"{run_date}_feature_metadata_class_clean.csv", index=False) # classified metadata, duplicates removed
# Settings and key numbers of this run; written last, so Pt2 only picks up complete runs
pd.DataFrame({
    "setting": ["TOX_DUST_FILE", "META_CLASS_FILE", "FIRST_FEAT_COL", "NAN_TO_ZERO", "NEG_HANDLING", "MIN_DETECT_FRAC",
                "MIN_UNIQUE_VALUES", "LOG_TRANSFORM_X", "GLOBAL_SEED", "n_samples", "n_endpoints",
                "n_dust_features_raw", "n_dust_features_kept"],
    "value": [TOX_DUST_FILE.name, META_CLASS_FILE.name, FIRST_FEAT_COL, NAN_TO_ZERO, NEG_HANDLING, MIN_DETECT_FRAC,
              MIN_UNIQUE_VALUES, LOG_TRANSFORM_X, GLOBAL_SEED, len(Y_tox_vs_dust), Y_tox_vs_dust.shape[1],
              X_tox_vs_dust.shape[1], X_tox_vs_dust_prep.shape[1]],
}).to_csv(OUT_DIR / f"{run_date}_preprocessing_settings.csv", index=False)

logging.info(f"Files for Pt2 (prefix {run_date}_): dust_detfilt_log10.csv, dust_raw_numeric.csv, "
             f"feature_metadata_class_clean.csv, preprocessing_settings.csv")
logging.info(f"Preprocessed files saved to: \n{OUT_DIR}")

minorsep()
logging.info(f"Done saving preprocessed dust & tox endpoints datasets (merged tox vs. dust dataset) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Preprocessing description
# Writes <yymmdd>_preprocessing_description.xlsx to OUT_DIR: one sheet per data type (dust, endpoints), one row per
# preprocessing step (Item, Explanation, Notebook cell), with this run's settings and numbers filled in
majorsep()
logging.info(f"Writing preprocessing description | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

from openpyxl.styles import Alignment, Font

def _g(name, default=None):
    """Value of a variable from this notebook run, or default if it does not exist."""
    return globals().get(name, default)

def _run(txt):
    """Adds ' In this run: ...' only if the numbers are available."""
    return f" In this run: {txt}" if txt else ""

def _fmt(n):
    return f"{n:,}" if isinstance(n, (int, np.integer)) else str(n)

n_smpl = len(_g("SAMPLES_MRGD", [])) or None
x_raw, x_prep = _g("X_tox_vs_dust"), _g("X_tox_vs_dust_prep")
n_raw = x_raw.shape[1] if x_raw is not None else None
n_kept = x_prep.shape[1] if x_prep is not None else None
labs = _g("filter_report")
lab_txt = ("; ".join(f"{lab}: {int(r['features_kept']):,} of {int(r['features_before']):,} kept"
                     for lab, r in labs.iterrows() if lab != "total") if labs is not None else "")
min_frac = _g("MIN_DETECT_FRAC", 0.10)
min_n = int(np.ceil(min_frac * n_smpl)) if n_smpl else None
neg = _g("NEG_HANDLING", "abs")
endpts = _g("TOX_ENDPTS_MRGD", [])
plan = _g("endpoint_info")
plan_txt = ("; ".join(f"{r['endpoint']}: {r['suggested_task']}" + (f" ({r['suggested_transform']})" if r["suggested_task"] == "regression" else "")
                      for _, r in plan.iterrows()) if plan is not None else "")
qc_tox = _g("tox_vs_dust_final_report")
miss_txt = ("; ".join(f"{r['endpoint']}: {int(r['missing'])}" for _, r in qc_tox.iterrows()) if qc_tox is not None else "")
rename = _g("TOX_COL_RENAME", {})
run_date = _g("run_date") or datetime.now().strftime("%y%m%d") # same prefix as the saved files

about_common = ("Created by the INQUIRE_TIP_Pt1_Preprocessing notebook (toxicity-driven prioritization of dust compounds). "
                "Steps are listed in the order they are applied. 'Notebook cell' = first line of the Pt1 cell (Pt2 for steps "
                "applied at modeling time).")

dust_rows = [
    ("ABOUT THIS FILE", f"Preprocessing of the dust chemical data (GCxGC-MS features: ID ending in _UMU; LC-MS features: ID ending in _VU or _VU_<n>) "
                        f"used as predictors in the models. {about_common}", ""),
    ("HOW TO READ", "Each row is one preprocessing step: what was done and why. Settings (variables in capitals) can be changed in the "
                    "notebook; their values for this run are given in the text. Numbers are filled in when the notebook writes this file.", ""),
    ("", "", ""),
    ("1. Input data", f"Merged tox vs. dust dataset ({_g('TOX_DUST_FILE').name if _g('TOX_DUST_FILE') else 'TOX_DUST_FILE'}): one row per dust sample; "
                      f"dust feature columns from '{_g('FIRST_FEAT_COL', '2427_0.69_UMU')}' to the last column. Values are concentrations "
                      f"(estimated or semi-quantitative / normalized peak areas, see 'concentration_type' in the metadata)."
                      + _run(f"{_fmt(n_raw)} dust features." if n_raw else ""), "# Load raw input data"),
    ("2. Column names", "Leading/trailing spaces removed from column names; repeated spaces collapsed to one. The run stops if this would create "
                        "duplicate column names.", "# Whitespace cleanup; # Harmonize column names"),
    ("3. Samples", "Rows without a sample ID dropped; for duplicated sample IDs only the first row is kept. Samples sorted by ID; the same samples "
                   "and order are used for the dust and endpoint data." + _run(f"{_fmt(n_smpl)} samples." if n_smpl else ""),
     "# Tox sample QC; # Create sample list; # Create dust predictor matrix"),
    ("4. Numeric values", "All dust columns converted to numbers; text entries (if any) set to missing (NaN).", "# Create dust predictor matrix"),
    ("5. Diagnostics", "Per feature: number of missing values, distinct values, detections (> 0), negative values, quantiles, skewness, "
                       "constant features (SD = 0), and the number of features per detection-frequency class. Diagnostics only, no changes to the data.",
     "# Dust matrix diagnostics; # Detection-frequency threshold check"),
    ("6. Missing values", f"Missing values (NaN) set to 0 = not detected (NAN_TO_ZERO = {_g('NAN_TO_ZERO', True)})."
                          + _run(f"{_fmt(_g('n_nan_dust'))} missing values set to 0." if _g("n_nan_dust") is not None else ""),
     "# Dust feature preprocessing"),
    ("7. Detection definition", "Detected = value > 0. Negative values mark concentrations below the limit of detection (LOD) and were reported "
                                "only for GCxGC-MS features (LC-MS non-detects are 0); they count as NOT detected for the detection-frequency filter."
                                + _run(f"{_fmt(_g('n_neg_dust'))} below-LOD values in {_fmt(_g('n_feat_neg'))} features." if _g("n_neg_dust") is not None else ""),
     "# Dust feature preprocessing"),
    ("8. Detection-frequency filter", f"Features kept if detected in at least MIN_DETECT_FRAC = {min_frac:.0%} of the samples"
                                      + (f" (≥ {min_n} of {n_smpl})" if min_n else "")
                                      + f" and with at least MIN_UNIQUE_VALUES = {_g('MIN_UNIQUE_VALUES', 5)} distinct values. Rarely detected "
                                      f"features cannot be linked reliably to toxicity with this number of samples and mainly add noise. The list of kept "
                                      f"features is saved in {run_date}_dust_feature_filter.csv."
                                      + _run(f"{_fmt(n_kept)} of {_fmt(n_raw)} features kept ({lab_txt})." if n_kept and n_raw else ""),
     "# Dust feature preprocessing"),
    ("9. Below-LOD values", (f"NEG_HANDLING = '{neg}': " + ("the minus sign is removed, i.e. the below-LOD values are kept as (less precise) "
                             "concentration estimates instead of being replaced by a constant." if neg == "abs" else
                             "below-LOD values set to 0 (not detected)."))
                            + _run(f"{_fmt(int(_g('neg_mask').sum().sum()))} below-LOD values in the kept features." if _g("neg_mask") is not None else ""),
     "# Dust feature preprocessing"),
    ("10. log10 transformation", "log10(x + c) per feature, with c = half of the smallest detected (originally positive) value of that feature. "
                                 "Concentrations span several orders of magnitude; without the log, a few very high samples dominate. Non-detects "
                                 "(0) all get the same lowest value log10(c), below every detection; below-LOD estimates stay close to that floor."
                                 + (f" Median feature skewness after the transformation: {np.nanmedian(_g('skew_after')):.2f}." if _g("skew_after") is not None else ""),
     "# Dust feature preprocessing"),
    ("11. Scaling", "Z-scores as in R scale(): (x − mean) / SD, with the sample SD (n − 1); features with SD = 0 dropped. In Pt2 the matrix is "
                    "scaled again on the samples available for each endpoint (samples with a missing endpoint value are left out), and "
                    "features that are constant within those samples are dropped."
                    + _run(f"{_fmt(len(_g('cols_to_drop_X')))} constant features dropped." if _g("cols_to_drop_X") is not None else ""),
     "# Scale dust predictor matrix; Pt2: # Endpoint selection"),
    ("12. Feature metadata", f"Classified feature metadata ({_g('META_CLASS_FILE').name if _g('META_CLASS_FILE') else 'META_CLASS_FILE'}): IDs as text "
                             f"without spaces; exact duplicate rows removed; for IDs with conflicting duplicates the first row is kept. "
                             f"In the merged lab metadata, '_<number>' suffixes are removed from InChIKeys.",
     "# Drop \"_<number>\" suffixes from InChIKeys; # Save preprocessed datasets"),
    ("13. Univariate screen (Pt2)", "The univariate screen in Pt2 uses the raw dust data (all features, NaN = 0, below-LOD handled as in step 9) "
                                    "with its own lower detection threshold, so rare compounds present mainly in the most active samples are not "
                                    "removed by the detection-frequency filter.", "Pt2: # Univariate screen"),
    ("14. Output files", f"{run_date}_dust_detfilt_log10.csv (modeling matrix: steps 1-10, not scaled); "
                         f"{run_date}_dust_detfilt_log10_scale().csv (steps 1-11, with metadata rows on top); "
                         f"{run_date}_dust_raw_numeric.csv (steps 1-4, all features); {run_date}_dust_feature_filter.csv (filter result per feature); "
                         f"{run_date}_feature_metadata_class_clean.csv and {run_date}_feature_metadata_dust_masked.csv (metadata of all / modeled features); "
                         f"{run_date}_preprocessing_settings.csv (settings read by Pt2).", "# Save preprocessed datasets"),
]

endpoint_rows = [
    ("ABOUT THIS FILE", f"Preprocessing of the toxicity endpoint data (bioassay results per dust sample) used as model responses. {about_common}", ""),
    ("HOW TO READ", "Each row is one preprocessing step: what was done and why. Step 0 was applied by the data provider before this pipeline "
                    "(as reported); all other steps by the notebooks. Numbers are filled in when the notebook writes this file.", ""),
    ("", "", ""),
    ("0. Before this pipeline (data provider)", "Values below the limit of quantification (<LOQ) replaced by LOQ/√2; outliers among the <LOQ values "
                                                "(IQR rule, applied to the values below LOQ) set to missing; negative values (for cell viability: cell "
                                                "proliferation) set to missing. Original values are not available to the pipeline.", ""),
    ("1. Input data", f"Merged tox vs. dust dataset: endpoint columns between the sample ID column and the first dust feature column"
                      + (f" ({len(endpts)} endpoints: {', '.join(endpts)})." if endpts else "."), "# Define tox endpoint columns"),
    ("2. Column names", "Raw endpoint names renamed to the standard names used in all outputs"
                        + (": " + "; ".join(f"{k} → {v}" for k, v in rename.items()) if rename else "") + ". Spaces cleaned as for the dust data.",
     "# Harmonize column names"),
    ("3. Samples", "Same sample handling as for the dust data (rows without ID dropped, first of duplicated IDs kept, sorted, aligned with the dust matrix)."
                   + _run(f"{_fmt(n_smpl)} samples." if n_smpl else ""), "# Tox sample QC; # Create response matrix"),
    ("4. Content check", "Per endpoint: counts of missing values, '<LOD' text, negative values, zeros, empty cells, 'not tested' entries and other "
                         "non-numeric entries (diagnostics only).", "# Raw tox content QC"),
    ("5. Numeric values", "Endpoint values converted to numbers; text entries set to missing (NaN). Missing values are NOT imputed."
                          + _run(f"missing values per endpoint: {miss_txt}." if miss_txt else ""), "# Tox QC summary"),
    ("6. Diagnostics", "Per endpoint: number of values, distinct values, quantiles, skewness, share of samples at the most common value.",
     "# Create tox matrix; # Final pre-ML tox endpoints QC"),
    ("7. Modeling plan", "Suggested per endpoint from its distribution: CLASSIFICATION (active vs. inactive; active = value above the lowest value) "
                         "if ≥ 35% of the samples share the lowest value (e.g. the <LOQ substitute or 0 = no effect) and both classes have ≥ 15 "
                         "samples; SKIP if one class is smaller; otherwise REGRESSION, on log10 scale if the skewness is > 1 and all values ≥ 0. "
                         f"Saved in {run_date}_endpoint_modeling_plan.csv. Can be overridden in Pt2 (TASK_OVERRIDE, FORCE_RUN)."
                         + _run(plan_txt + "." if plan_txt else ""), "# Final pre-ML tox endpoints QC"),
    ("8. Missing values at modeling (Pt2)", "Samples with a missing value for the modeled endpoint are left out for that endpoint only, so each "
                                            "endpoint uses all its available samples.", "Pt2: # Endpoint selection"),
    ("9. Target transformation (Pt2)", "Regression: log10(y), or log10(y + c) with c = half the smallest positive value if zeros occur; no "
                                       "transformation for endpoints that are not skewed. Classification: 1 = value above the lowest value "
                                       "(above the median if no block of values sits at the minimum), 0 = otherwise.", "Pt2: # Endpoint selection"),
    ("10. Output files", f"{run_date}_tox_endpoints_unscaled.csv (steps 1-5; transformations are applied in Pt2); "
                         f"{run_date}_endpoint_modeling_plan.csv (step 7).", "# Save preprocessed datasets"),
]

def _sheet_df(rows):
    return pd.DataFrame(rows, columns=["Item", "Explanation", "Notebook cell"])

desc_file = OUT_DIR / f"{run_date}_preprocessing_description.xlsx"
with pd.ExcelWriter(desc_file, engine="openpyxl") as writer:
    for sheet, rows in [("dust", dust_rows), ("endpoints", endpoint_rows)]:
        _sheet_df(rows).to_excel(writer, sheet_name=sheet, index=False)
        ws = writer.sheets[sheet]
        for col, width in zip("ABC", [34, 120, 42]):
            ws.column_dimensions[col].width = width
        for row in ws.iter_rows(min_row=1):
            for c in row:
                c.alignment = Alignment(wrap_text=True, vertical="top")
        for c in ws[1]:
            c.font = Font(bold=True)
        ws.freeze_panes = "A2"

logging.info(f"Saved to: \n{desc_file}")
minorsep()
logging.info(f"Done writing preprocessing description | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
